In [4]:
import os
import sys
import subprocess
import ctypes
import sys as sys_lib

PROJECT_ROOT = '/Users/doanvinhnhan/Roo-Code'

if os.getcwd() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# 2. Import thư viện cốt lõi của Manim
from manim import *

# 3. Import toàn bộ thư viện custom của bạn

from skills.fami_lib import *
from skills.fami_assets_helper import *
from skills.fami_effects import * # Import thêm file này phòng trường hợp các hiệu ứng nằm ở đây
from skills.bit import BitSequence
from skills.broadcasting import Broadcasting
from skills.receiving import Receiving
config.tex_template = TexTemplate()
config.tex_template.add_to_preamble(r"\usepackage[utf8]{vietnam}")

TexTemplate(_body='', tex_compiler='latex', description='', output_format='.dvi', documentclass='\\documentclass[preview]{standalone}', preamble='\\usepackage[english]{babel}\n\\usepackage{amsmath}\n\\usepackage{amssymb}\n\\usepackage[utf8]{vietnam}', placeholder_text='YourTextHere', post_doc_commands='')

In [2]:
%%manim -v ERROR --resolution 1080,1920 --frame_rate 60 --media_dir /Users/doanvinhnhan/Roo-Code/media Scene1_Hook

class Scene1_Hook(FaMIBaseScene):
    def construct(self):
        title = self.create_title("TIN NHẮN DỄ BỊ NHIỄU SÓNG?", "LÀM SAO ĐỂ ĐO LƯỜNG?")
        
        # Mobjects
        tin_nhan = load_svg_icon("assets/Nhieu_Song/Hook/tin-nhan.svg")
        if tin_nhan:
            tin_nhan.set_stroke(width=4)
            tin_nhan.scale(1.0).move_to(LEFT * 2.0 + UP * 0.5)
        
        may_bao = load_svg_icon("assets/Nhieu_Song/Hook/may-bao.svg")
        if may_bao:
            may_bao.set_stroke(width=4)
            may_bao.scale(1.2).move_to(RIGHT * 1.5 + UP * 0.5)
        
        clock = load_svg_icon("assets/Nhieu_Song/Hook/clock.svg")
        if clock:
            clock.set_stroke(width=4)
            clock.scale(1.5).move_to(UP * 0.5)
        
        with self.voiceover(text="Tin nhắn gửi đi rất dễ bị nhiễu sóng.") as tracker:
            self.update_subtitle("Tin nhắn gửi đi rất dễ bị nhiễu sóng.")
            self.play(Write(title), run_time=min(1.0, tracker.duration * 0.3))
            
            if tin_nhan:
                anim, r_func = skill_pop_in(tin_nhan)
                self.play(anim, rate_func=r_func, run_time=min(0.5, tracker.duration * 0.2))
            
            if may_bao and tin_nhan:
                self.play(
                    FadeIn(may_bao, shift=DOWN), 
                    Wiggle(tin_nhan, scale_value=1.2, rotation_angle=0.1 * PI),
                    run_time=min(1.0, tracker.duration * 0.3)
                )
            
        with self.voiceover(text="Vậy lấy gì để đo lường chất lượng đường truyền?") as tracker:
            self.update_subtitle("Vậy lấy gì để đo lường chất lượng đường truyền?")
            
            anims_out = []
            if tin_nhan: anims_out.append(FadeOut(tin_nhan, shift=LEFT))
            if may_bao: anims_out.append(FadeOut(may_bao, shift=UP))
            if anims_out:
                self.play(*anims_out, run_time=min(0.5, tracker.duration * 0.2))
            
            if clock:
                anim, r_func = skill_pop_in(clock)
                self.play(anim, rate_func=r_func, run_time=min(0.8, tracker.duration * 0.4))
                
                # Sóng tỏa ra
                arcs = VGroup(*[
                    Arc(radius=1.5 + i*0.6, angle=PI/2, start_angle=PI/4, color=SUCCESS, stroke_width=8)
                    for i in range(3)
                ])
                arcs.rotate(PI/4)
                arcs.next_to(clock, UP, buff=0.3)
                
                self.play(Create(arcs, lag_ratio=0.5), run_time=min(1.0, tracker.duration * 0.2))

        self.finish_scene()


Manim Community v0.19.0

[03/24/26 17:55:18] WARNING  Some options were not used: {'shortest': '1', 'metadata':     ]8;id=380592;file:///opt/anaconda3/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=395824;file:///opt/anaconda3/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#801\801]8;;\
                             'comment=Rendered with Manim Community v0.19.0'}                                      

In [ ]:
%%manim -v ERROR --resolution 720,1280 --frame_rate 30 --media_dir /Users/doanvinhnhan/Roo-Code/media mainScene
config.tex_template = TexTemplate()
config.tex_template.add_to_preamble(r"\usepackage[utf8]{vietnam}")
config.tex_template.add_to_preamble(r"\usepackage{enumitem}")
config.tex_template.add_to_preamble(r"\usepackage{xcolor}")
from scipy import stats
import math

class mainScene(FaMIBaseScene):
    def construct(self):

        #1. Setup + subscene 1
        grid = NumberPlane(
            x_range=[-8, 8, 1],
            y_range=[-4.5, 4.5, 1],
            background_line_style={
                "stroke_color": TEAL,
                "stroke_width": 2,
                "stroke_opacity": 0.3
            },
            axis_config={
                "stroke_width":0,
                "include_tip":False,
            }
        )
        text_0 = Tex(r"BEP\\(Bit Error Probability)"
        )
        text_1 = Tex(r"Xác suất lỗi bit, được định nghĩa là xác suất để một bit bất kỳ bị sai trạng thái (từ 0 thành 1 hoặc ngược lại) sau khi đi qua kênh truyền",
                 font_size=24)
        title = Tex(r"Xác suất lỗi Bit").move_to(UP*4.8)

        self.play(FadeIn(grid))
        self.play(FadeIn(text_0))
        self.play(text_0.animate.shift(UP))
        self.play(Write(text_1))
        self.play(FadeOut(VGroup(text_0,text_1)))
        self.play(FadeIn(title))
        self.wait(3)

        #2. Setup + subscene 2


        broadcasting = Broadcasting().scale(0.35)
        receiving = Receiving().scale(0.35)
        self.arrange_comparison(broadcasting, receiving)
        VGroup(broadcasting, receiving).shift(UP * 1.0)
        self.play(FadeIn(broadcasting), FadeIn(receiving))

        orig_bits = "10110010"
        recv_bits_str = "00100110"
        m_bits = BitSequence(sequence=orig_bits, color="#00FF00", scale_factor=0.45)
        
        broadcasting.update_sequence(self, orig_bits, scale_factor=0.45)
        m_bits.next_to(broadcasting.icon, RIGHT)
        self.play(FadeIn(m_bits))
        self.play(m_bits.animate.next_to(receiving.icon, LEFT))
        receiving.update_sequence(self, orig_bits, recv_bits_str, scale_factor=0.45)
        self.play(FadeOut(m_bits))
        self.wait(3)


        #3. Subscene 3
        self.play(VGroup(broadcasting, receiving).animate.move_to(UP * 2))

        bit_sent = broadcasting.current_seq_display.copy()
        bit_get = receiving.current_seq_display.copy()
        text_0 = Text("Có 3 bit bị lỗi", t2c={"3":RED})
        text_1 = Tex(r"BER(Bit error rate) = $\frac{3}{8} = 0.375$", font_size =48)

    
        self.play(bit_sent.animate.move_to(UP*0).scale(2))
        self.play(bit_get.animate.move_to(UP*-0.5).scale(2))
        self.play(FadeIn(text_0.move_to(UP*-1.5)))
        self.play(Write(text_1.move_to(UP*-3)))


        self.play(FadeOut(VGroup(bit_sent, bit_get, text_0, text_1)))
        self.wait(3)

        #4. Subscene 4

        line1_1 = Tex(
            "Việc truyền tải ", "$N$", " bit dữ liệu có thể mô phỏng thành dãy", 
            font_size=30
        )
        line1_2 = Tex("$N$", " phép thử độc lập.",
            font_size=30
        )

        line2_1 = Tex(
            "Gọi ", "$X_i$", " là biến ngẫu nhiên chỉ trạng thái của bit thứ ", "$i$,", 
            font_size=30
        )
        line2_2 = Tex("$X_i \\in \\{0,1\\}$", ", trong đó:",
            font_size=30
        )
        # Sử dụng \bullet để mô phỏng itemize
        line3 = Tex(
            r"$\bullet$ ", "$X_i = 1$", " là sự kiện bit thứ $i$ ", "truyền đúng", 
            font_size=30
        )
        line4 = Tex(
            r"$\bullet$ ", "$X_i = 0$", " là sự kiện bit thứ $i$ ", "truyền sai", 
            font_size=30
        )

        line1_1[1].set_color(YELLOW)
        line1_2[0].set_color(YELLOW)
        line1_2[1].set_color(YELLOW)

        line2_1[1].set_color(BLUE)
        line2_1[3].set_color(YELLOW)
        line2_2[0].set_color(BLUE)

        line3[1].set_color("#00FF00")
        line3[3].set_color("#00FF00")
        line4[1].set_color(RED)   
        line4[3].set_color(RED) 

        paragraph = VGroup(line1_1, line1_2, line2_1, line2_2, line3, line4).arrange(DOWN, aligned_edge=LEFT).shift(DOWN*1)
        
        line3.shift(RIGHT * 0.5)
        line4.shift(RIGHT * 0.5)

        for line in paragraph:
            self.play(Write(line), run_time=1.5)
            self.wait(0.5)

        self.wait(3)

        #5. Subscene 5
        self.play(FadeOut(VGroup(paragraph,broadcasting, receiving)))


        text_0 = Tex(r"Khi đó,", font_size = 30)
        equation_0 = MathTex(r"X_i \sim \text{Bernoulli}(\text{BEP})")
        text_1 = Tex(r"Và", font_size = 30)
        equation_1 = MathTex(
            r"\text{BER} &\sim B(N, \text{BEP}) \\",
        ) 
        equation_2 = MathTex(
            r"\text{BEP} &\approx \lim\limits_{N\to \infty} \text{BER} = \lim\limits_{N\to \infty} \sum_{i = 1}^{N}X_i"

        )       
        n, p = 20, 0.5
        x_values = np.arange(0, n + 1)
        y_values = [stats.binom.pmf(k, n, p) for k in x_values]

        B_chart = BarChart(
            values=y_values,
            x_length=10,
            y_length=6,
            y_range=[0, 0.2, 0.05],
            axis_config={"font_size": 24},
            bar_colors = [BLUE],
            bar_fill_opacity = 0.8,
            x_axis_config={
                "include_ticks": False,       
            },
            y_axis_config={
                "stroke_opacity": 0,          
                "include_ticks": False, 
                "include_numbers": False 
            },
        )
        if B_chart.get_y_axis().numbers:
            B_chart.get_y_axis().numbers.set_opacity(0)

        p = 0.2
        x_values = [1 - p, p]  
        
        Bernoulli_chart = BarChart(
            values=x_values,
            x_length=4,          
            y_length=10,         
            y_range=[0, 1, 0.2], 
            axis_config={"font_size": 24},
            bar_colors=[BLUE, BLUE],
            bar_fill_opacity=0.8,
            x_axis_config={
                "include_ticks": False,
            },
            y_axis_config={
                "stroke_opacity": 0,          
                "include_ticks": False, 
                "include_numbers": False 
            },
        ).rotate(-PI/2)
        if Bernoulli_chart.get_y_axis().numbers:
            Bernoulli_chart.get_y_axis().numbers.set_opacity(0)

        content_1 = VGroup(text_0, equation_0, Bernoulli_chart).arrange(DOWN, buff=0.5)
        text_0.align_to(equation_2, LEFT)
        Bernoulli_chart.scale(0.3).shift(UP*1.5)

        self.play(Write(content_1))
        self.play(content_1.animate.shift(UP*1))

        content_2 = VGroup(text_1, equation_1, B_chart).arrange(DOWN, buff=0.5).shift(DOWN*3.75)
        text_1.align_to(equation_2, LEFT)
        B_chart.scale(0.3).shift(UP*1.8)
        
        self.play(Write(content_2))

        self.play(FadeOut(VGroup(content_1, content_2)))

        self.wait(3)

        self.play(Write(equation_2))
        self.wait(3)
        self.play(FadeOut(equation_2))

        #6. Subscene 6
        
        num_packets = 50000
        error_prob = 0.15
        
        backend_dir = os.path.join(PROJECT_ROOT, "scripts", "backend")
        exe_path = os.path.join(backend_dir, "generate_packet")
        total_bits = num_packets * 8 
        run_cmd = [exe_path, str(total_bits), str(error_prob)]
        result = subprocess.run(run_cmd, capture_output=True, text=True, check=True)
        output_lines = result.stdout.strip().split("\n")
        
        full_orig_bits_str = output_lines[0]
        full_recv_bits_str = output_lines[1]

        val_tracker = ValueTracker(2)
        head_pos_container = [ORIGIN]
        
        _text_cache_main = {}
        def get_cached_text(text_str, font_size, color=WHITE, font=""):
            key = (str(text_str), font_size, color, font)
            if key not in _text_cache_main:
                kwargs = {"font_size": font_size, "weight": BOLD}
                if font:
                    kwargs["font"] = font
                t = Text(str(text_str), **kwargs).set_color(color)
                _text_cache_main[key] = t
            return _text_cache_main[key].copy()

        lib_ext = ".dylib" if sys_lib.platform == "darwin" else ".so"
        lib_path = os.path.join(backend_dir, f"libsimulate{lib_ext}")
        ber_lib = ctypes.CDLL(lib_path)
        ber_lib.calculate_ber_array_from_str.argtypes = [ctypes.c_char_p, ctypes.c_char_p, ctypes.c_int, ctypes.c_int, ctypes.POINTER(ctypes.c_int)]
        ber_lib.calculate_ber_array_from_str.restype = ctypes.POINTER(ctypes.c_double)
        ber_lib.free_ber_array.argtypes = [ctypes.POINTER(ctypes.c_double)]
        ber_lib.free_ber_array.restype = None
        
        out_array_size = ctypes.c_int(0)
        c_orig_bits = full_orig_bits_str.encode('utf-8')
        c_recv_bits = full_recv_bits_str.encode('utf-8')
        
        ber_array_ptr = ber_lib.calculate_ber_array_from_str(c_orig_bits, c_recv_bits, total_bits, 8, ctypes.byref(out_array_size))
        precomputed_ber_values = [ber_array_ptr[i] for i in range(out_array_size.value)]
        ber_lib.free_ber_array(ber_array_ptr)
            

        broadcasting = Broadcasting().scale(0.35)
        receiving = Receiving().scale(0.35)
        self.arrange_comparison(broadcasting, receiving, buff=1.5)
        VGroup(broadcasting, receiving).shift(UP * 3)
        text_0 = Text("Mô phỏng Monte Carlo").move_to(UP*4.8)
        self.play(Transform(title,text_0))
        self.play(FadeIn(VGroup(broadcasting, receiving)))

        def get_plot():
            val = max(val_tracker.get_value(), 2) * 8
            step = val / 8
            
            y_max = 0.5 + 0.8 * math.exp(-(val - 2) / 50.0)
            y_step = y_max / 5
            
            axes = Axes(
                x_range=[0, val, step],
                y_range=[0, y_max, y_step],
                x_length=7.5,
                y_length=4,
                axis_config={
                    "stroke_color": GREY_A,
                    "include_numbers": False,
                }
            )
            
            x_values = np.linspace(0, val, 6)
            for x in x_values:
                x_str = str(int(x))
                t = get_cached_text(x_str, 16)
                t.next_to(axes.c2p(x, 0), DOWN, buff=0.1)
                axes.add(t)

            y_values = np.linspace(0, y_max, 6)
            for y in y_values:
                y_str = f"{y:.2f}"
                t = get_cached_text(y_str, 16)
                t.next_to(axes.c2p(0, y), LEFT, buff=0.1)
                axes.add(t)
            
            current_x_max = int(val)
            # stride = 5 để chống ngập RAM khi render Manim CE dynamic
            stride = 100 
            indices = list(range(0, current_x_max + 1, stride))
            if indices[-1] != current_x_max:
                indices.append(current_x_max)
                
            labels = axes.get_axis_labels(x_label=Text("N", font_size=20), y_label=Text("BER", font_size=20))
            target_line = axes.plot(lambda x: 0.15, color=GREY, stroke_width=2).set_opacity(0.5)

            points = []
            for x in indices:
                i_val = min(x // 8, len(precomputed_ber_values) - 1)
                points.append(axes.c2p(x, precomputed_ber_values[i_val]))
            
            graph = VMobject(color=YELLOW, stroke_width=4)
            graph.set_points_as_corners(points)
            
            vgroup = VGroup(axes, labels, target_line, graph).move_to(DOWN * 0.5).scale(0.9)
            head_pos_container[0] = points[-1]
            return vgroup

        dynamic_plot = always_redraw(get_plot)
        self.add(dynamic_plot)
        
        def get_dynamic_ber_text():
            val = max(val_tracker.get_value(), 2)
            current_idx = min(int(val), len(precomputed_ber_values)-1)
            current_ber = precomputed_ber_values[current_idx]
            ber_str = f"BER = {current_ber:.4f}"
            text = get_cached_text(ber_str, 24)
            text.next_to(head_pos_container[0] + UP*0.5 + LEFT*2)
            return text

        dynamic_ber = always_redraw(get_dynamic_ber_text)
        self.add(dynamic_ber)
        # --- FAST FORWARD CHO CÁC GÓI CHẠY LIÊN TỤC TRÊN MÀN HÌNH NHƯ BẢN GỐC ---
        if broadcasting.current_seq_display:
            self.remove(broadcasting.current_seq_display)
        if hasattr(broadcasting, 'prev_seq_display') and broadcasting.prev_seq_display:
            self.remove(broadcasting.prev_seq_display)
        if receiving.current_seq_display:
            self.remove(receiving.current_seq_display)
            
        # Để tránh lỗi Aborted (OOM memory leak củ Manim), ta không tạo bù nhìn vô hạn Mobject nữa
        # Mà thay đổi nội dung (become) của một lượng text giới hạn
        b_text = BitSequence(sequence="00000000", color="#00FF00", font_size=36, scale_factor=0.27)
        b_text.next_to(broadcasting.label, DOWN, buff=0.25)
        
        prev_b_text = BitSequence(sequence="00000000", color="#00FF00", font_size=36, scale_factor=0.27)
        prev_b_text.next_to(b_text, DOWN, buff=0.15)
        prev_b_text.set_opacity(0.3)
        
        r_text = VGroup()
        for _ in range(8):
            r_text.add(get_cached_text("0", 36, "#00FF00"))
        r_text.arrange(RIGHT, buff=0.1).scale(0.27)
        r_text.next_to(receiving.label, DOWN, buff=0.25)
        
        m_bits = BitSequence(sequence="00000000", color="#00FF00", font_size=36, scale_factor=0.27)
        
        self.add(b_text, prev_b_text, r_text, m_bits)
        
        last_packet_idx = [-1]
        
        def transmission_updater(mob, dt):
            val = val_tracker.get_value()
            if val >= num_packets:
                return
            packet_idx = int(val)
            progress = val - packet_idx
            
            if packet_idx != last_packet_idx[0]:
                last_packet_idx[0] = packet_idx
                start_idx = packet_idx * 8
                end_idx = start_idx + 8
                orig_bits = full_orig_bits_str[start_idx:end_idx]
                recv_bits = full_recv_bits_str[start_idx:end_idx]
                
                # Cập nhật thay vì clear
                new_b_text = BitSequence(sequence=orig_bits, color="#00FF00", font_size=36, scale_factor=0.27)
                new_b_text.move_to(b_text)
                b_text.become(new_b_text)
                
                if packet_idx > 0:
                    prev_start_idx = (packet_idx - 1) * 8
                    prev_orig_bits = full_orig_bits_str[prev_start_idx:prev_start_idx+8]
                    new_prev_b = BitSequence(sequence=prev_orig_bits, color="#00FF00", font_size=36, scale_factor=0.27)
                    new_prev_b.move_to(prev_b_text)
                    new_prev_b.set_opacity(0.3)
                    prev_b_text.become(new_prev_b)
                    
                for k in range(8):
                    if k < len(orig_bits):
                        c = "#00FF00" if orig_bits[k] == recv_bits[k] else RED
                        new_t = get_cached_text(recv_bits[k], 36 * 0.27, c)
                        new_t.move_to(r_text[k])
                        r_text[k].become(new_t)

            start_idx = packet_idx * 8
            orig_bits = full_orig_bits_str[start_idx:start_idx+8]
            new_m_bits = BitSequence(sequence=orig_bits, color="#00FF00", font_size=36, scale_factor=0.27)
            
            start_pos = broadcasting.icon.get_right() + RIGHT*0.2
            end_pos = receiving.icon.get_left() + LEFT*0.2
            current_pos = interpolate(start_pos, end_pos, progress)
            new_m_bits.move_to(current_pos)
            m_bits.become(new_m_bits)

        m_bits.add_updater(transmission_updater)
        self.play(
            val_tracker.animate.set_value(num_packets), 
            run_time=6.0, 
            rate_func=lambda t: 0.05 * t + 0.95 * (t**5)
        )
        m_bits.remove_updater(transmission_updater)
        self.remove(m_bits, b_text, prev_b_text, r_text)

        self.wait(3)

        #7. Subscene 7

        self.play(FadeOut(VGroup(broadcasting, receiving, dynamic_plot, dynamic_ber)))

        data =  [
            ["8",f"{precomputed_ber_values[7]:.4f}"],
            ["64",f"{precomputed_ber_values[63]:.4f}"],
            ["...","..."],
            ["32768",f"{precomputed_ber_values[32767]:.4f}"],
            ["400000",f"{precomputed_ber_values[-1]:.4f}"],
        ]
        table = Table(
            data,
            col_labels = [Tex(r"$N$"), Tex(r"BER")],
            include_outer_lines=True
        )
        table_hlines = table.get_horizontal_lines()
        table_vlines = table.get_vertical_lines()
        table_entries = table.get_entries()
        table.scale(0.6).shift(UP*1.5)
        self.play(Create(table_hlines), Create(table_vlines), run_time=2)
        self.wait(1)
        self.play(FadeIn(table_entries), lag_ratio=0.1) 
        self.wait(2)

        text_0 = Tex(r"$\Rightarrow$ BEP $\approx 0.15$", font_size=48).move_to(DOWN*2.25)
        self.play(Write(text_0))

        self.play(FadeOut(VGroup(grid, text_0, table)))





Manim Community v0.19.0

In [195]:
%%manim -v ERROR --resolution 720,1280 --frame_rate 30 --media_dir /Users/doanvinhnhan/Roo-Code/media testScene
config.tex_template = TexTemplate()
config.tex_template.add_to_preamble(r"\usepackage[utf8]{vietnam}")
config.tex_template.add_to_preamble(r"\usepackage{enumitem}")
config.tex_template.add_to_preamble(r"\usepackage{xcolor}")
import math

class testScene(FaMIBaseScene):
    def construct(self):
        # Thiết lập tốc độ di chuyển và số lượng frame gửi
        run_time_speed = 1.6
        num_packets = 50000
        error_prob = 0.15
        sum_eq = MathTex(r"E = \sum_{i=1}^{N} X_i \sim \text{Binomial}(N, p)", font_size=40).move_to(DOWN * 1.0)

        
        # 0. Generate Bits from Backend C++
        import subprocess
        import ctypes
        import sys as sys_lib
        
        backend_dir = os.path.join(PROJECT_ROOT, "scripts", "backend")
        exe_path = os.path.join(backend_dir, "generate_packet")
        
        # 1000 gói
        total_bits = num_packets * 8 
        run_cmd = [exe_path, str(total_bits), str(error_prob)]
        result = subprocess.run(run_cmd, capture_output=True, text=True, check=True)
        output_lines = result.stdout.strip().split("\n")
        
        full_orig_bits_str = output_lines[0]
        full_recv_bits_str = output_lines[1]

        
        broadcasting = Broadcasting().scale(0.7)
        receiving = Receiving().scale(0.7)
        self.arrange_comparison(broadcasting, receiving)
        VGroup(broadcasting, receiving).shift(UP * 1.0)

        # --- SETUP TRACKER & CACHING TEXT ---
        val_tracker = ValueTracker(2)
        head_pos_container = [ORIGIN]
        
        _text_cache_main = {}
        def get_cached_text(text_str, font_size, color=WHITE, font=""):
            key = (str(text_str), font_size, color, font)
            if key not in _text_cache_main:
                kwargs = {"font_size": font_size, "weight": BOLD}
                if font:
                    kwargs["font"] = font
                t = Text(str(text_str), **kwargs).set_color(color)
                _text_cache_main[key] = t
            return _text_cache_main[key].copy()

        with self.voiceover(text="Theo Luật số lớn, khi kích thước mẫu N tiến tới vô cùng, trung bình mẫu sẽ hội tụ theo xác suất về giá trị kỳ vọng lý thuyết.") as tracker:
            self.update_subtitle("Theo Luật số lớn, khi N tiến tới vô cùng,\ntrung bình mẫu sẽ hội tụ về kỳ vọng lý thuyết.")
            
            self.play(FadeOut(sum_eq), run_time=min(0.5, tracker.duration * 0.1))
            
            sys_group = VGroup(broadcasting, receiving)
            self.play(
                sys_group.animate.scale(0.6).to_edge(UP, buff=1.5), 
                run_time=min(1.0, tracker.duration * 0.3)
            )
            
            # Không tạo static block nữa, chờ dynamic plot
            # Lấy data đã sinh ở đầu hàm
            lib_ext = ".dylib" if sys_lib.platform == "darwin" else ".so"
            lib_path = os.path.join(backend_dir, f"libsimulate{lib_ext}")
            ber_lib = ctypes.CDLL(lib_path)
            ber_lib.calculate_ber_array_from_str.argtypes = [ctypes.c_char_p, ctypes.c_char_p, ctypes.c_int, ctypes.c_int, ctypes.POINTER(ctypes.c_int)]
            ber_lib.calculate_ber_array_from_str.restype = ctypes.POINTER(ctypes.c_double)
            ber_lib.free_ber_array.argtypes = [ctypes.POINTER(ctypes.c_double)]
            ber_lib.free_ber_array.restype = None
            
            out_array_size = ctypes.c_int(0)
            c_orig_bits = full_orig_bits_str.encode('utf-8')
            c_recv_bits = full_recv_bits_str.encode('utf-8')
            
            ber_array_ptr = ber_lib.calculate_ber_array_from_str(c_orig_bits, c_recv_bits, total_bits, 8, ctypes.byref(out_array_size))
            precomputed_ber_values = [ber_array_ptr[i] for i in range(out_array_size.value)]
            ber_lib.free_ber_array(ber_array_ptr)
            
            def get_plot():
                val = max(val_tracker.get_value(), 2)
                step = val / 5
                
                y_max = 0.5 + 0.8 * math.exp(-(val - 2) / 50.0)
                y_step = y_max / 5
                
                axes = Axes(
                    x_range=[0, val, step],
                    y_range=[0, y_max, y_step],
                    x_length=7.5,
                    y_length=4,
                    axis_config={
                        "stroke_color": GREY_A,
                        "include_numbers": False,
                    }
                )
                
                x_values = np.linspace(0, val, 6)
                for x in x_values:
                    x_str = str(int(x))
                    t = get_cached_text(x_str, 16)
                    t.next_to(axes.c2p(x, 0), DOWN, buff=0.1)
                    axes.add(t)

                y_values = np.linspace(0, y_max, 6)
                for y in y_values:
                    y_str = f"{y:.2f}"
                    t = get_cached_text(y_str, 16)
                    t.next_to(axes.c2p(0, y), LEFT, buff=0.1)
                    axes.add(t)
                
                current_x_max = int(val)
                # stride = 5 để chống ngập RAM khi render Manim CE dynamic
                stride = 5 
                indices = list(range(0, current_x_max + 1, stride))
                if indices[-1] != current_x_max:
                    indices.append(current_x_max)
                    
                labels = axes.get_axis_labels(x_label=Text("N", font_size=20), y_label=Text("BER", font_size=20))
                target_line = axes.plot(lambda x: 0.15, color=GREY, stroke_width=2).set_opacity(0.5)

                points = []
                for x in indices:
                    i_val = min(x, len(precomputed_ber_values) - 1)
                    points.append(axes.c2p(x, precomputed_ber_values[i_val]))
                
                graph = VMobject(color=YELLOW, stroke_width=4)
                graph.set_points_as_corners(points)
                
                vgroup = VGroup(axes, labels, target_line, graph).move_to(DOWN * 0.5).scale(0.9)
                head_pos_container[0] = points[-1]
                return vgroup

            dynamic_plot = always_redraw(get_plot)
            self.add(dynamic_plot)
            
            def get_dynamic_ber_text():
                val = max(val_tracker.get_value(), 2)
                current_idx = min(int(val), len(precomputed_ber_values)-1)
                current_ber = precomputed_ber_values[current_idx]
                ber_str = f"BER = {current_ber:.4f}"
                text = get_cached_text(ber_str, 24)
                text.next_to(head_pos_container[0] + UP*0.5 + LEFT*2)
                return text

            dynamic_ber = always_redraw(get_dynamic_ber_text)
            self.add(dynamic_ber)
            
            # --- FAST FORWARD CHO CÁC GÓI CHẠY LIÊN TỤC TRÊN MÀN HÌNH NHƯ BẢN GỐC ---
            if broadcasting.current_seq_display:
                self.remove(broadcasting.current_seq_display)
            if hasattr(broadcasting, 'prev_seq_display') and broadcasting.prev_seq_display:
                self.remove(broadcasting.prev_seq_display)
            if receiving.current_seq_display:
                self.remove(receiving.current_seq_display)
                
            # Để tránh lỗi Aborted (OOM memory leak củ Manim), ta không tạo bù nhìn vô hạn Mobject nữa
            # Mà thay đổi nội dung (become) của một lượng text giới hạn
            b_text = BitSequence(sequence="00000000", color="#00FF00", font_size=36, scale_factor=0.27)
            b_text.next_to(broadcasting.label, DOWN, buff=0.25)
            
            prev_b_text = BitSequence(sequence="00000000", color="#00FF00", font_size=36, scale_factor=0.27)
            prev_b_text.next_to(b_text, DOWN, buff=0.15)
            prev_b_text.set_opacity(0.3)
            
            r_text = VGroup()
            for _ in range(8):
                r_text.add(get_cached_text("0", 36, "#00FF00"))
            r_text.arrange(RIGHT, buff=0.1).scale(0.27)
            r_text.next_to(receiving.label, DOWN, buff=0.25)
            
            m_bits = BitSequence(sequence="00000000", color="#00FF00", font_size=36, scale_factor=0.27)
            
            self.add(b_text, prev_b_text, r_text, m_bits)
            
            last_packet_idx = [-1]
            
            def transmission_updater(mob, dt):
                val = val_tracker.get_value()
                if val >= num_packets:
                    return
                packet_idx = int(val)
                progress = val - packet_idx
                
                if packet_idx != last_packet_idx[0]:
                    last_packet_idx[0] = packet_idx
                    start_idx = packet_idx * 8
                    end_idx = start_idx + 8
                    orig_bits = full_orig_bits_str[start_idx:end_idx]
                    recv_bits = full_recv_bits_str[start_idx:end_idx]
                    
                    # Cập nhật thay vì clear
                    new_b_text = BitSequence(sequence=orig_bits, color="#00FF00", font_size=36, scale_factor=0.27)
                    new_b_text.move_to(b_text)
                    b_text.become(new_b_text)
                    
                    if packet_idx > 0:
                        prev_start_idx = (packet_idx - 1) * 8
                        prev_orig_bits = full_orig_bits_str[prev_start_idx:prev_start_idx+8]
                        new_prev_b = BitSequence(sequence=prev_orig_bits, color="#00FF00", font_size=36, scale_factor=0.27)
                        new_prev_b.move_to(prev_b_text)
                        new_prev_b.set_opacity(0.3)
                        prev_b_text.become(new_prev_b)
                        
                    for k in range(8):
                        if k < len(orig_bits):
                            c = "#00FF00" if orig_bits[k] == recv_bits[k] else RED
                            new_t = get_cached_text(recv_bits[k], 36 * 0.27, c)
                            new_t.move_to(r_text[k])
                            r_text[k].become(new_t)

                start_idx = packet_idx * 8
                orig_bits = full_orig_bits_str[start_idx:start_idx+8]
                new_m_bits = BitSequence(sequence=orig_bits, color="#00FF00", font_size=36, scale_factor=0.27)
                
                start_pos = broadcasting.icon.get_right() + RIGHT*0.2
                end_pos = receiving.icon.get_left() + LEFT*0.2
                current_pos = interpolate(start_pos, end_pos, progress)
                new_m_bits.move_to(current_pos)
                m_bits.become(new_m_bits)

            m_bits.add_updater(transmission_updater)
            self.play(
                val_tracker.animate.set_value(num_packets), 
                run_time=min(6.0, tracker.duration * 0.5), 
                rate_func=lambda t: 0.05 * t + 0.95 * (t**5)
            )
            m_bits.remove_updater(transmission_updater)
            self.remove(m_bits, b_text, prev_b_text, r_text)


Manim Community v0.19.0

[03/25/26 10:33:02] WARNING  Some options were not used: {'shortest': '1', 'metadata':     ]8;id=337143;file:///opt/anaconda3/lib/python3.12/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=81053;file:///opt/anaconda3/lib/python3.12/site-packages/manim/scene/scene_file_writer.py#801\801]8;;\
                             'comment=Rendered with Manim Community v0.19.0'}                                      

In [ ]:
%%manim -v ERROR --resolution 720,1280 --frame_rate 30 --media_dir /Users/doanvinhnhan/Roo-Code/media class mainScenewithvoice(FaMIBaseScene):

config.tex_template = TexTemplate()
config.tex_template.add_to_preamble(r"\usepackage[utf8]{vietnam}")
config.tex_template.add_to_preamble(r"\usepackage{enumitem}")
config.tex_template.add_to_preamble(r"\usepackage{xcolor}")
from scipy import stats
import math

class mainScenewithvoice(FaMIBaseScene):
    def construct(self):

        #1. Setup + subscene 1
        grid = NumberPlane(
            x_range=[-8, 8, 1],
            y_range=[-4.5, 4.5, 1],
            background_line_style={
                "stroke_color": TEAL,
                "stroke_width": 2,
                "stroke_opacity": 0.3
            },
            axis_config={
                "stroke_width":0,
                "include_tip":False,
            }
        )
        text_0 = Tex(r"BEP\\(Bit Error Probability)"
        )
        text_1 = Tex(r"Xác suất lỗi bit, được định nghĩa là xác suất để một bit bất kỳ bị sai trạng thái (từ 0 thành 1 hoặc ngược lại) sau khi đi qua kênh truyền",
                 font_size=24)
        title = Tex(r"Xác suất lỗi Bit").move_to(UP*4.8)

        self.play(FadeIn(grid))
        self.play(FadeIn(text_0))
        self.play(text_0.animate.shift(UP))
        self.play(Write(text_1))
        self.play(FadeOut(VGroup(text_0,text_1)))
        self.play(FadeIn(title))
        self.wait(3)

        #2. Setup + subscene 2


        broadcasting = Broadcasting().scale(0.35)
        receiving = Receiving().scale(0.35)
        self.arrange_comparison(broadcasting, receiving)
        VGroup(broadcasting, receiving).shift(UP * 1.0)
        self.play(FadeIn(broadcasting), FadeIn(receiving))

        orig_bits = "10110010"
        recv_bits_str = "00100110"
        m_bits = BitSequence(sequence=orig_bits, color="#00FF00", scale_factor=0.45)
        
        broadcasting.update_sequence(self, orig_bits, scale_factor=0.45)
        m_bits.next_to(broadcasting.icon, RIGHT)
        self.play(FadeIn(m_bits))
        self.play(m_bits.animate.next_to(receiving.icon, LEFT))
        receiving.update_sequence(self, orig_bits, recv_bits_str, scale_factor=0.45)
        self.play(FadeOut(m_bits))
        self.wait(3)


        #3. Subscene 3
        self.play(VGroup(broadcasting, receiving).animate.move_to(UP * 2))

        bit_sent = broadcasting.current_seq_display.copy()
        bit_get = receiving.current_seq_display.copy()
        text_0 = Text("Có 3 bit bị lỗi", t2c={"3":RED})
        text_1 = Tex(r"BER(Bit error rate) = $\frac{3}{8} = 0.375$", font_size =48)

    
        self.play(bit_sent.animate.move_to(UP*0).scale(2))
        self.play(bit_get.animate.move_to(UP*-0.5).scale(2))
        self.play(FadeIn(text_0.move_to(UP*-1.5)))
        self.play(Write(text_1.move_to(UP*-3)))


        self.play(FadeOut(VGroup(bit_sent, bit_get, text_0, text_1)))
        self.wait(3)

        #4. Subscene 4

        line1_1 = Tex(
            "Việc truyền tải ", "$N$", " bit dữ liệu có thể mô phỏng thành dãy", 
            font_size=30
        )
        line1_2 = Tex("$N$", " phép thử độc lập.",
            font_size=30
        )

        line2_1 = Tex(
            "Gọi ", "$X_i$", " là biến ngẫu nhiên chỉ trạng thái của bit thứ ", "$i$,", 
            font_size=30
        )
        line2_2 = Tex("$X_i \\in \\{0,1\\}$", ", trong đó:",
            font_size=30
        )
        # Sử dụng \bullet để mô phỏng itemize
        line3 = Tex(
            r"$\bullet$ ", "$X_i = 1$", " là sự kiện bit thứ $i$ ", "truyền đúng", 
            font_size=30
        )
        line4 = Tex(
            r"$\bullet$ ", "$X_i = 0$", " là sự kiện bit thứ $i$ ", "truyền sai", 
            font_size=30
        )

        line1_1[1].set_color(YELLOW)
        line1_2[0].set_color(YELLOW)
        line1_2[1].set_color(YELLOW)

        line2_1[1].set_color(BLUE)
        line2_1[3].set_color(YELLOW)
        line2_2[0].set_color(BLUE)

        line3[1].set_color("#00FF00")
        line3[3].set_color("#00FF00")
        line4[1].set_color(RED)   
        line4[3].set_color(RED) 

        paragraph = VGroup(line1_1, line1_2, line2_1, line2_2, line3, line4).arrange(DOWN, aligned_edge=LEFT).shift(DOWN*1)
        
        line3.shift(RIGHT * 0.5)
        line4.shift(RIGHT * 0.5)

        for line in paragraph:
            self.play(Write(line), run_time=1.5)
            self.wait(0.5)

        self.wait(3)

        #5. Subscene 5
        self.play(FadeOut(VGroup(paragraph,broadcasting, receiving)))


        text_0 = Tex(r"Khi đó,", font_size = 30)
        equation_0 = MathTex(r"X_i \sim \text{Bernoulli}(\text{BEP})")
        text_1 = Tex(r"Và", font_size = 30)
        equation_1 = MathTex(
            r"\text{BER} &\sim B(N, \text{BEP}) \\",
        ) 
        equation_2 = MathTex(
            r"\text{BEP} &\approx \lim\limits_{N\to \infty} \text{BER} = \lim\limits_{N\to \infty} \sum_{i = 1}^{N}X_i"

        )       
        n, p = 20, 0.5
        x_values = np.arange(0, n + 1)
        y_values = [stats.binom.pmf(k, n, p) for k in x_values]

        B_chart = BarChart(
            values=y_values,
            x_length=10,
            y_length=6,
            y_range=[0, 0.2, 0.05],
            axis_config={"font_size": 24},
            bar_colors = [BLUE],
            bar_fill_opacity = 0.8,
            x_axis_config={
                "include_ticks": False,       
            },
            y_axis_config={
                "stroke_opacity": 0,          
                "include_ticks": False, 
                "include_numbers": False 
            },
        )
        if B_chart.get_y_axis().numbers:
            B_chart.get_y_axis().numbers.set_opacity(0)

        p = 0.2
        x_values = [1 - p, p]  
        
        Bernoulli_chart = BarChart(
            values=x_values,
            x_length=4,          
            y_length=10,         
            y_range=[0, 1, 0.2], 
            axis_config={"font_size": 24},
            bar_colors=[BLUE, BLUE],
            bar_fill_opacity=0.8,
            x_axis_config={
                "include_ticks": False,
            },
            y_axis_config={
                "stroke_opacity": 0,          
                "include_ticks": False, 
                "include_numbers": False 
            },
        ).rotate(-PI/2)
        if Bernoulli_chart.get_y_axis().numbers:
            Bernoulli_chart.get_y_axis().numbers.set_opacity(0)

        content_1 = VGroup(text_0, equation_0, Bernoulli_chart).arrange(DOWN, buff=0.5)
        text_0.align_to(equation_2, LEFT)
        Bernoulli_chart.scale(0.3).shift(UP*1.5)

        self.play(Write(content_1))
        self.play(content_1.animate.shift(UP*1))

        content_2 = VGroup(text_1, equation_1, B_chart).arrange(DOWN, buff=0.5).shift(DOWN*3.75)
        text_1.align_to(equation_2, LEFT)
        B_chart.scale(0.3).shift(UP*1.8)
        
        self.play(Write(content_2))

        self.play(FadeOut(VGroup(content_1, content_2)))

        self.wait(3)

        self.play(Write(equation_2))
        self.wait(3)
        self.play(FadeOut(equation_2))

        #6. Subscene 6
        
        num_packets = 50000
        error_prob = 0.15
        
        backend_dir = os.path.join(PROJECT_ROOT, "scripts", "backend")
        exe_path = os.path.join(backend_dir, "generate_packet")
        total_bits = num_packets * 8 
        run_cmd = [exe_path, str(total_bits), str(error_prob)]
        result = subprocess.run(run_cmd, capture_output=True, text=True, check=True)
        output_lines = result.stdout.strip().split("\n")
        
        full_orig_bits_str = output_lines[0]
        full_recv_bits_str = output_lines[1]

        val_tracker = ValueTracker(2)
        head_pos_container = [ORIGIN]
        
        _text_cache_main = {}
        def get_cached_text(text_str, font_size, color=WHITE, font=""):
            key = (str(text_str), font_size, color, font)
            if key not in _text_cache_main:
                kwargs = {"font_size": font_size, "weight": BOLD}
                if font:
                    kwargs["font"] = font
                t = Text(str(text_str), **kwargs).set_color(color)
                _text_cache_main[key] = t
            return _text_cache_main[key].copy()

        lib_ext = ".dylib" if sys_lib.platform == "darwin" else ".so"
        lib_path = os.path.join(backend_dir, f"libsimulate{lib_ext}")
        ber_lib = ctypes.CDLL(lib_path)
        ber_lib.calculate_ber_array_from_str.argtypes = [ctypes.c_char_p, ctypes.c_char_p, ctypes.c_int, ctypes.c_int, ctypes.POINTER(ctypes.c_int)]
        ber_lib.calculate_ber_array_from_str.restype = ctypes.POINTER(ctypes.c_double)
        ber_lib.free_ber_array.argtypes = [ctypes.POINTER(ctypes.c_double)]
        ber_lib.free_ber_array.restype = None
        
        out_array_size = ctypes.c_int(0)
        c_orig_bits = full_orig_bits_str.encode('utf-8')
        c_recv_bits = full_recv_bits_str.encode('utf-8')
        
        ber_array_ptr = ber_lib.calculate_ber_array_from_str(c_orig_bits, c_recv_bits, total_bits, 8, ctypes.byref(out_array_size))
        precomputed_ber_values = [ber_array_ptr[i] for i in range(out_array_size.value)]
        ber_lib.free_ber_array(ber_array_ptr)
            

        broadcasting = Broadcasting().scale(0.35)
        receiving = Receiving().scale(0.35)
        self.arrange_comparison(broadcasting, receiving, buff=1.5)
        VGroup(broadcasting, receiving).shift(UP * 3)
        text_0 = Text("Mô phỏng Monte Carlo").move_to(UP*4.8)
        self.play(Transform(title,text_0))
        self.play(FadeIn(VGroup(broadcasting, receiving)))

        def get_plot():
            val = max(val_tracker.get_value(), 2) * 8
            step = val / 8
            
            y_max = 0.5 + 0.8 * math.exp(-(val - 2) / 50.0)
            y_step = y_max / 5
            
            axes = Axes(
                x_range=[0, val, step],
                y_range=[0, y_max, y_step],
                x_length=7.5,
                y_length=4,
                axis_config={
                    "stroke_color": GREY_A,
                    "include_numbers": False,
                }
            )
            
            x_values = np.linspace(0, val, 6)
            for x in x_values:
                x_str = str(int(x))
                t = get_cached_text(x_str, 16)
                t.next_to(axes.c2p(x, 0), DOWN, buff=0.1)
                axes.add(t)

            y_values = np.linspace(0, y_max, 6)
            for y in y_values:
                y_str = f"{y:.2f}"
                t = get_cached_text(y_str, 16)
                t.next_to(axes.c2p(0, y), LEFT, buff=0.1)
                axes.add(t)
            
            current_x_max = int(val)
            # stride = 5 để chống ngập RAM khi render Manim CE dynamic
            stride = 100 
            indices = list(range(0, current_x_max + 1, stride))
            if indices[-1] != current_x_max:
                indices.append(current_x_max)
                
            labels = axes.get_axis_labels(x_label=Text("N", font_size=20), y_label=Text("BER", font_size=20))
            target_line = axes.plot(lambda x: 0.15, color=GREY, stroke_width=2).set_opacity(0.5)

            points = []
            for x in indices:
                i_val = min(x // 8, len(precomputed_ber_values) - 1)
                points.append(axes.c2p(x, precomputed_ber_values[i_val]))
            
            graph = VMobject(color=YELLOW, stroke_width=4)
            graph.set_points_as_corners(points)
            
            vgroup = VGroup(axes, labels, target_line, graph).move_to(DOWN * 0.5).scale(0.9)
            head_pos_container[0] = points[-1]
            return vgroup

        dynamic_plot = always_redraw(get_plot)
        self.add(dynamic_plot)
        
        def get_dynamic_ber_text():
            val = max(val_tracker.get_value(), 2)
            current_idx = min(int(val), len(precomputed_ber_values)-1)
            current_ber = precomputed_ber_values[current_idx]
            ber_str = f"BER = {current_ber:.4f}"
            text = get_cached_text(ber_str, 24)
            text.next_to(head_pos_container[0] + UP*0.5 + LEFT*2)
            return text

        dynamic_ber = always_redraw(get_dynamic_ber_text)
        self.add(dynamic_ber)
        # --- FAST FORWARD CHO CÁC GÓI CHẠY LIÊN TỤC TRÊN MÀN HÌNH NHƯ BẢN GỐC ---
        if broadcasting.current_seq_display:
            self.remove(broadcasting.current_seq_display)
        if hasattr(broadcasting, 'prev_seq_display') and broadcasting.prev_seq_display:
            self.remove(broadcasting.prev_seq_display)
        if receiving.current_seq_display:
            self.remove(receiving.current_seq_display)
            
        # Để tránh lỗi Aborted (OOM memory leak củ Manim), ta không tạo bù nhìn vô hạn Mobject nữa
        # Mà thay đổi nội dung (become) của một lượng text giới hạn
        b_text = BitSequence(sequence="00000000", color="#00FF00", font_size=36, scale_factor=0.27)
        b_text.next_to(broadcasting.label, DOWN, buff=0.25)
        
        prev_b_text = BitSequence(sequence="00000000", color="#00FF00", font_size=36, scale_factor=0.27)
        prev_b_text.next_to(b_text, DOWN, buff=0.15)
        prev_b_text.set_opacity(0.3)
        
        r_text = VGroup()
        for _ in range(8):
            r_text.add(get_cached_text("0", 36, "#00FF00"))
        r_text.arrange(RIGHT, buff=0.1).scale(0.27)
        r_text.next_to(receiving.label, DOWN, buff=0.25)
        
        m_bits = BitSequence(sequence="00000000", color="#00FF00", font_size=36, scale_factor=0.27)
        
        self.add(b_text, prev_b_text, r_text, m_bits)
        
        last_packet_idx = [-1]
        
        def transmission_updater(mob, dt):
            val = val_tracker.get_value()
            if val >= num_packets:
                return
            packet_idx = int(val)
            progress = val - packet_idx
            
            if packet_idx != last_packet_idx[0]:
                last_packet_idx[0] = packet_idx
                start_idx = packet_idx * 8
                end_idx = start_idx + 8
                orig_bits = full_orig_bits_str[start_idx:end_idx]
                recv_bits = full_recv_bits_str[start_idx:end_idx]
                
                # Cập nhật thay vì clear
                new_b_text = BitSequence(sequence=orig_bits, color="#00FF00", font_size=36, scale_factor=0.27)
                new_b_text.move_to(b_text)
                b_text.become(new_b_text)
                
                if packet_idx > 0:
                    prev_start_idx = (packet_idx - 1) * 8
                    prev_orig_bits = full_orig_bits_str[prev_start_idx:prev_start_idx+8]
                    new_prev_b = BitSequence(sequence=prev_orig_bits, color="#00FF00", font_size=36, scale_factor=0.27)
                    new_prev_b.move_to(prev_b_text)
                    new_prev_b.set_opacity(0.3)
                    prev_b_text.become(new_prev_b)
                    
                for k in range(8):
                    if k < len(orig_bits):
                        c = "#00FF00" if orig_bits[k] == recv_bits[k] else RED
                        new_t = get_cached_text(recv_bits[k], 36 * 0.27, c)
                        new_t.move_to(r_text[k])
                        r_text[k].become(new_t)

            start_idx = packet_idx * 8
            orig_bits = full_orig_bits_str[start_idx:start_idx+8]
            new_m_bits = BitSequence(sequence=orig_bits, color="#00FF00", font_size=36, scale_factor=0.27)
            
            start_pos = broadcasting.icon.get_right() + RIGHT*0.2
            end_pos = receiving.icon.get_left() + LEFT*0.2
            current_pos = interpolate(start_pos, end_pos, progress)
            new_m_bits.move_to(current_pos)
            m_bits.become(new_m_bits)

        m_bits.add_updater(transmission_updater)
        self.play(
            val_tracker.animate.set_value(num_packets), 
            run_time=6.0, 
            rate_func=lambda t: 0.05 * t + 0.95 * (t**5)
        )
        m_bits.remove_updater(transmission_updater)
        self.remove(m_bits, b_text, prev_b_text, r_text)

        self.wait(3)

        #7. Subscene 7

        self.play(FadeOut(VGroup(broadcasting, receiving, dynamic_plot, dynamic_ber)))

        data =  [
            ["8",f"{precomputed_ber_values[7]:.4f}"],
            ["64",f"{precomputed_ber_values[63]:.4f}"],
            ["...","..."],
            ["32768",f"{precomputed_ber_values[32767]:.4f}"],
            ["400000",f"{precomputed_ber_values[-1]:.4f}"],
        ]
        table = Table(
            data,
            col_labels = [Tex(r"$N$"), Tex(r"BER")],
            include_outer_lines=True
        )
        table_hlines = table.get_horizontal_lines()
        table_vlines = table.get_vertical_lines()
        table_entries = table.get_entries()
        table.scale(0.6).shift(UP*1.5)
        self.play(Create(table_hlines), Create(table_vlines), run_time=2)
        self.wait(1)
        self.play(FadeIn(table_entries), lag_ratio=0.1) 
        self.wait(2)

        text_0 = Tex(r"$\Rightarrow$ BEP $\approx 0.15$", font_size=48).move_to(DOWN*2.25)
        self.play(Write(text_0))

        self.play(FadeOut(VGroup(grid, text_0, table)))



